# Leaflet cluster map of talk locations

Scrapes the `location` YAML front-matter field from every `.md` file inside `_talks/`,
geolocates each one with `geopy/Nominatim`, and uses the `getorg` library to write a
standalone Leaflet cluster map into `talkmap/`.

**Designed to be non-fatal for CI use**:
- Imports are wrapped and never crash the notebook
- Files missing `title` / `venue` / `location` fall back to safe defaults or are skipped
- Nominatim service errors (very common on shared GitHub Actions IPs) are printed as warnings
- `getorg` map generation is wrapped and never raises back to `nbconvert`


In [ ]:
try:
    import frontmatter
    import glob
    import getorg
    from geopy import Nominatim
    from geopy.exc import (
        GeocoderTimedOut,
        GeocoderServiceError,
        GeocoderQuotaExceeded,
        GeocoderUnavailable,
    )
    import traceback, sys, os, time
    print("[OK] All imports loaded")
except Exception as _e:
    print(f"[FAIL] Import error: {_e}")
    import traceback; traceback.print_exc()
    frontmatter = None; glob = None; getorg = None; Nominatim = None


In [ ]:
# Collect the Markdown files
try:
    g = sorted(glob.glob("_talks/*.md"))
    print(f"Found {len(g)} talk file(s): {g}")
except Exception as ex:
    print(f"[FAIL] Could not glob _talks/*.md: {ex}")
    g = []


In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 10

geocoder = None
location_dict = {}
location = ""
permalink = ""
title = ""

try:
    # Use a user-agent that is more likely to work in CI environments
    geocoder = Nominatim(user_agent="henu-innopv-ci/1.0", timeout=TIMEOUT)
    print("[OK] Nominatim geocoder initialized")
except Exception as ex:
    print(f"[WARN] Could not initialize Nominatim geocoder: {ex}")
    print("       (geolocation step will be skipped)")
    geocoder = None


> All failures in the next cell are caught, reported, and skipped.
> Nominatim commonly rate-limits or refuses shared GitHub Actions IP addresses — that is expected and non-fatal.


In [ ]:
# Perform geolocation — NON-FATAL: any failure per talk is caught and skipped
location_dict = {}
failed_files = []

for file in g:
    print(f"\n--- Processing {file} ---")

    # 1) Parse frontmatter safely
    try:
        data = frontmatter.load(file)
        data = data.to_dict()
    except Exception as ex:
        print(f"  SKIP: failed to parse frontmatter: {ex}")
        failed_files.append((file, "frontmatter", str(ex)))
        continue

    # 2) Skip when no location field
    if "location" not in data or not str(data.get("location", "")).strip():
        print("  SKIP: no location field — bypassing")
        continue

    # 3) Safe field access (title/venue may be missing)
    title   = str(data.get("title", "")).strip() or "(untitled)"
    venue   = str(data.get("venue", "")).strip() or "(no venue)"
    location = str(data["location"]).strip()
    description = title + "<br />" + venue + "; " + location

    # 4) If geocoder is unavailable, still record with None so the pipeline still runs
    if geocoder is None:
        print("  SKIP: geocoder unavailable (see earlier warning)")
        failed_files.append((file, "no-geocoder", location))
        continue

    # 5) Geocode — wrap all Nominatim exceptions
    try:
        result = geocoder.geocode(location, timeout=TIMEOUT)
        if result is None:
            print(f"  WARN: Nominatim returned None for location='{location}'")
            failed_files.append((file, "geocode-None", location))
        else:
            location_dict[description] = result
            print(f"  OK: {location} -> ({result.latitude}, {result.longitude})")
    except (GeocoderTimedOut, GeocoderServiceError, GeocoderQuotaExceeded, GeocoderUnavailable) as ex:
        print(f"  WARN: Nominatim service error for '{location}': {type(ex).__name__}: {ex}")
        failed_files.append((file, "geocoder-service", f"{type(ex).__name__}: {ex}"))
    except Exception as ex:
        print(f"  WARN: geocode unexpected error for '{location}': {type(ex).__name__}: {ex}")
        failed_files.append((file, "geocode-other", f"{type(ex).__name__}: {ex}"))

    # Nominatim usage policy requires max 1 request/second
    time.sleep(1.1)

print("\n========== SUMMARY ==========")
print(f"Successful geolocations: {len(location_dict)}")
print(f"Failed / skipped:         {len(failed_files)}")
for _f, _reason, _msg in failed_files:
    print(f"  - {_f} : {_reason} : {_msg}")


In [ ]:
# Save the map — NON-FATAL: getorg calls are wrapped and never raise to nbconvert
print(f"\n--- Saving cluster map with {len(location_dict)} locations ---")
try:
    m = getorg.orgmap.create_map_obj()
    getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)
    print("[OK] output_html_cluster_map completed")
    if os.path.isdir("talkmap"):
        print("talkmap/ contents:", sorted(os.listdir("talkmap")))
    else:
        print("[WARN] talkmap/ directory was not created by getorg")
except Exception as ex:
    print(f"[FAIL] getorg output error: {type(ex).__name__}: {ex}")
    import traceback; traceback.print_exc()
    print("(continuing — failures captured above; no map written)")


In [ ]:
# End of pipeline
print("\nDone.")
